In [ ]:
import os
import sys 
os.chdir("/workspaces/dev/app")
sys.path.append("/workspaces/dev/app")

In [ ]:
from services.whisper import WordService, WordParams, WordReturn
from services.whisper.Params import Hyperparameters
import librosa
import numpy as np
from silero_vad import load_silero_vad, get_speech_timestamps

In [ ]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [ ]:
audio, sr = librosa.load("/workspaces/dev/.data/news_with_english.mp3", sr=SAMPLE_RATE)

In [ ]:
# audio = audio[85 * SAMPLE_RATE:]

In [ ]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=32000, scale=400))
  rand_len = np.clip(rand_len, 28000, 36000)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [ ]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [ ]:
# print(full_text)

In [ ]:
HYPERPARAMETERS = {
  "weighted_prob_boundary": 0,
  "filter_by_duration_z": {
    "default": 2.0,
    "ko": 1.5,
    "en": 2.0,
  },
  "filter_by_probability": {
    "z": {
      "default": 2.0,
      "ko": 2.0,
      "en": 2.0,
    },
    "min_prob": {
      "default": 1.0,
      "ko": 0.4,
      "en": 0.4,
    },
  },
  "token_iou_padding": 0.2,
  "combine": {
    "search_range_time": {
      "default": 1.5,
      "ko": 1.5,
      "en": 1.5,
    },
    "threshold": {
      "default": 0.5,
      "ko": 0.25,
      "en": 0.5,
    },
    "tolerance": {
      "default": 0.3,
      "ko": 0.3,
      "en": 0.3,
    },
  },
  "refine_tolerance": {
    "default": 0.5,
    "ko": 0.5,
    "en": 0.5,
  }
}

In [ ]:
MAX_PREV_TIME = 5

In [ ]:
class TestWordService(WordService):
  def _get_weighted_probability(self, probabilities, start, end, duration, boundary):
    center = (start + end) / 2
    if center > boundary:
      if center < duration - boundary:
        return probabilities
      return probabilities - probabilities * ((boundary - duration + center)/boundary) ** 2
    return probabilities - probabilities * (boundary - center/boundary) ** 2

In [ ]:
hyper = Hyperparameters(None, HYPERPARAMETERS)

In [ ]:
whisper_service = TestWordService.get_instance(MAX_PREV_TIME, hyper)

In [ ]:
model = load_silero_vad(onnx=True)

In [ ]:
raise Exception("stop")

In [ ]:
def vad_audio(audio):
  timestamps = get_speech_timestamps(
    audio,
    model,
    sampling_rate=SAMPLE_RATE,
    threshold=0.4,
    min_silence_duration_ms = 400,
    speech_pad_ms=300
  )

  merged_audio = []

  for segment in timestamps:
    start = segment['start']
    end = segment['end']
    merged_audio.append(audio[start:end])

  return np.concatenate(merged_audio)


In [ ]:
from IPython.display import Audio

In [ ]:
segment_id = 0
completed = {}
word_params = WordParams()

In [ ]:
segment = segments[segment_id]
segment_id += 1

word_params.audio = vad_audio(segment)

(result, audio) = whisper_service.transcribe(word_params)
completed.update(result.completed_dict)

print(f"{segment_id}" + "--" * 20)
print([(v.lang, v.text) for k, v in completed.items()])
print([(v.lang, v.text) for v in result.prev_words if v.is_word])
print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

word_params.order = result.order
word_params.time_offset = result.time_offset
word_params.prev_audio = result.prev_audio
word_params.prev_words = result.prev_words
word_params.prev_recog = result.prev_recog
word_params.prev_prob_mean = result.prev_prob_mean
word_params.prev_prob_std = result.prev_prob_std
word_params.prev_prob_count = result.prev_prob_count
word_params.prev_dura_mean = result.prev_dura_mean
word_params.prev_dura_std = result.prev_dura_std
word_params.prev_dura_count = result.prev_dura_count

Audio(audio, rate=SAMPLE_RATE)

In [ ]:
Audio(result.prev_audio, rate=SAMPLE_RATE)

In [ ]:
for segment in segments:
  word_params.audio = segment

  (result, audio) = whisper_service.transcribe(word_params)
  completed.update(result.completed_dict)

  print(f"{segment_id}" + "--" * 20)
  print([(v.lang, v.text) for k, v in completed.items()])
  print([(v.lang, v.text) for v in result.prev_words if v.is_word])
  print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

  word_params.order = result.order
  word_params.time_offset = result.time_offset
  word_params.prev_audio = result.prev_audio
  word_params.prev_words = result.prev_words
  word_params.prev_recog = result.prev_recog
  word_params.prev_prob_mean = result.prev_prob_mean
  word_params.prev_prob_std = result.prev_prob_std
  word_params.prev_prob_count = result.prev_prob_count
  word_params.prev_dura_mean = result.prev_dura_mean
  word_params.prev_dura_std = result.prev_dura_std
  word_params.prev_dura_count = result.prev_dura_count

In [ ]:
for key, item in completed.items():
  print(key, item)
for v in result.prev_words:
  if v.is_word: print(v.text)
# for v in prev_recog:
#   print(v.text)